# RAG Agent validation (dev)
Este notebook es utilizado para realizar la validación del RAG antes de hacer el deploy en el serving endpoint

> Nota: en Compute Serverless suele faltar `mlflow`; por eso instalamos dependencias en la primera celda.


In [0]:
# # # ==============================
# # # 1) Dependencias (solo para pruebas locales)
# # # ==============================
# # # En Serverless compute puede no venir mlflow instalado.
%pip install -U mlflow
%pip install -U databricks-vectorsearch mlflow databricks-sdk
dbutils.library.restartPython()


In [0]:
# ==============================
# 2) Config (ajustar a tu workspace)
# ==============================
from pathlib import Path
import time

# --- Vector Search ---
VS_ENDPOINT = "bind_agent_vs"
VS_INDEX_FULL_NAME = "bind_agent.docs.pdf_chunks_vs_idx"

# --- Endpoints ---
EMBED_ENDPOINT = "databricks-bge-large-en"  # for query_vector
EMB_ENDPOINT = EMBED_ENDPOINT  # backward-compatible alias
# LLM_ENDPOINT = "databricks-gemma-3-12b"
LLM_ENDPOINT = 'databricks-llama-4-maverick' 

# --- Retrieval params ---
TOP_K_CANDIDATES = "40"
LEX_FALLBACK_LIMIT = "20"
# TOP_K_CANDIDATES = "120"
# LEX_FALLBACK_LIMIT = "60"
TOP_K_FINAL = "8"

# --- Prompt/context limits ---
MAX_CONTEXT_CHARS = "14000"
RERANK_SNIPPET_CHARS = "1200"

# --- LLM params ---
TEMPERATURE_RERANK = "0.0"
TEMPERATURE_ANSWER = "0.2"
MAX_TOKENS_ANSWER = "900"

# --- Retry params ---
MAX_RETRIES = "3"
RETRY_SLEEP_SECS = "1.0"

# MLflow / UC
# Recomendación: catalog.schema.model
UC_MODEL_NAME = "bind_agent.docs.rag_agent"

# Experimento: usar ruta en /Users/... para evitar problemas de permisos
current_user = spark.sql("select current_user() as u").first()["u"]
EXPERIMENT_PATH = f"/Users/{current_user}/bind_agent/rag_agent_deploy"

# Serving endpoint para el agente
MODEL_SERVING_ENDPOINT = "bind_agent_rag_agent"


In [0]:
import os, importlib.util
from pathlib import Path

# 0) Setear env vars usadas en el RAG
os.environ["RAG_LLM_ENDPOINT"] = LLM_ENDPOINT
os.environ["RAG_EMBED_ENDPOINT"] = EMBED_ENDPOINT
os.environ["RAG_VS_ENDPOINT"] = VS_ENDPOINT
os.environ["RAG_VS_INDEX"] = VS_INDEX_FULL_NAME
os.environ["RAG_VS_INDEX_FULL_NAME"] = VS_INDEX_FULL_NAME
os.environ["RAG_TOP_K_CANDIDATES"] = TOP_K_CANDIDATES
os.environ["RAG_TOP_K_FINAL"] = TOP_K_FINAL
os.environ["RAG_LEX_FALLBACK_LIMIT"] = LEX_FALLBACK_LIMIT
os.environ["RAG_MAX_CONTEXT_CHARS"] = MAX_CONTEXT_CHARS
os.environ["RAG_RERANK_SNIPPET_CHARS"] = RERANK_SNIPPET_CHARS
os.environ["RAG_TEMPERATURE_RERANK"] = TEMPERATURE_RERANK
os.environ["RAG_TEMPERATURE_ANSWER"] = TEMPERATURE_ANSWER
os.environ["RAG_MAX_TOKENS_ANSWER"] = MAX_TOKENS_ANSWER
os.environ["RAG_MAX_RETRIES"] = MAX_RETRIES
os.environ["RAG_RETRY_SLEEP_SECS"] = RETRY_SLEEP_SECS
# os.environ["DATABRICKS_TOKEN"] = token # Acceso para el service endpoint al search index 
# os.environ["DATABRICKS_HOST"] = host # Acceso para el service endpoint al search index 

# env usadas en el serving endpoint
env_vars = {
    "RAG_LLM_ENDPOINT": os.environ["RAG_LLM_ENDPOINT"],
    "RAG_EMBED_ENDPOINT": os.environ["RAG_EMBED_ENDPOINT"],
    "RAG_VS_ENDPOINT": os.environ["RAG_VS_ENDPOINT"],
    "RAG_VS_INDEX": os.environ["RAG_VS_INDEX"],
    "RAG_VS_INDEX_FULL_NAME": os.environ["RAG_VS_INDEX_FULL_NAME"],
    "RAG_TOP_K_CANDIDATES": os.environ["RAG_TOP_K_CANDIDATES"],
    "RAG_TOP_K_FINAL": os.environ["RAG_TOP_K_FINAL"],
    "RAG_LEX_FALLBACK_LIMIT": os.environ["RAG_LEX_FALLBACK_LIMIT"],
    "RAG_MAX_CONTEXT_CHARS": os.environ["RAG_MAX_CONTEXT_CHARS"],
    "RAG_RERANK_SNIPPET_CHARS": os.environ["RAG_RERANK_SNIPPET_CHARS"],
    "RAG_TEMPERATURE_RERANK": os.environ["RAG_TEMPERATURE_RERANK"],
    "RAG_TEMPERATURE_ANSWER": os.environ["RAG_TEMPERATURE_ANSWER"],
    "RAG_MAX_TOKENS_ANSWER": os.environ["RAG_MAX_TOKENS_ANSWER"],
    "RAG_MAX_RETRIES": os.environ["RAG_MAX_RETRIES"],
    "RAG_RETRY_SLEEP_SECS": os.environ["RAG_RETRY_SLEEP_SECS"],
    # "DATABRICKS_TOKEN": os.environ["DATABRICKS_TOKEN"],
    # "DATABRICKS_HOST": os.environ["DATABRICKS_HOST"],
}

Reinicio del cache para que se tomen los cambios en el RAG

In [0]:
import sys, importlib

def unload_rag_modules():
    prefixes = ("rag_agent", "rag_core", "rag_lib", "bind_rag_agent")
    for m in list(sys.modules.keys()):
        if m in prefixes or any(m.startswith(p + ".") for p in prefixes):
            sys.modules.pop(m, None)
    importlib.invalidate_caches()

unload_rag_modules()

Variables para activar el trace

In [0]:
# # 1) Activar trace
os.environ["RAG_DEBUG_TRACE"] = "1"
os.environ["RAG_DEBUG_TRACE_ALL"] = "1"   # si querés todos los candidatos

Preguntas que parsa el smoketest

In [0]:
## Si no tiene comentarios al lado es porque se está contestando correctamente

# query = '¿Cuál fue la contribución de cada segmento al resultado de octubre 2025?'
query = '¿Cuál fue resultado operativo de cada segmento para octubre 2025?'
# query = "¿Cuál fue la contribución de cada segmento al resultado de octubre 2025?"
# query = "¿Cuál fue el resultado neto de octubre 2025?"
# query = "¿Cuál fue el resultado neto de mayo de 2025?"
# query = "¿Cual fue el gasto directo en enero de 2025?"
# query = "Dame el resultado neto del cliente grimoldi para julio de 2025"



SMOKE TEST: Respuesta curada

In [0]:
import importlib.util
from pathlib import Path
from textwrap import shorten
import sys

# --- 1) Cargar el módulo ---
agent_py = Path("/Workspace/Users/emanuel.vieira@sunnydata.ai/bind_agent/src/bind_rag_agent/rag_agent.py")
sys.path.insert(0, str(agent_py.parent.parent))
print(sys.path[0])

spec = importlib.util.spec_from_file_location("rag_agent_module", str(agent_py))
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

# --- 2) Helpers de pretty print ---
def _pp_result(res: dict, max_hits: int = 5, max_chars: int = 280):
    print("\n" + "="*88)
    print("QUERY:", res.get("query",""))
    print("-"*88)
    print("ANSWER:\n")
    print(res.get("answer","").strip())
    print("-"*88)

    cands = res.get("retrieved_candidates") or []
    reranked = res.get("reranked_hits") or []

    print(f"Retrieved candidates: {len(cands)} | Reranked hits: {len(reranked)}")

    hits = reranked if reranked else cands
    if not hits:
        print("\n(No hay hits para mostrar)")
        return

    print(f"\nTOP {min(max_hits, len(hits))} EVIDENCIAS:")
    for i, h in enumerate(hits[:max_hits], 1):
        file_date = h.get("file_date") or ""
        path = h.get("path") or ""
        page = h.get("page_num")
        topic = h.get("topic") or ""
        cid = h.get("chunk_id") or ""

        loc = []
        if file_date: loc.append(file_date)
        if path: loc.append(path)
        if page is not None: loc.append(f"p.{page}")
        if topic: loc.append(f"topic={topic}")
        loc = " | ".join(loc)

        print(f"\n[{i}] {loc}")
        print(f"    chunk_id: {cid}")

def _pp_minimal(res: dict):
    # por si querés una versión ultra corta
    print("\n" + "="*88)
    print(res.get("answer","").strip())
    print("="*88)


In [0]:
res = mod.answer_with_rag(query)

# Elegí una:
_pp_result(res, max_hits=8, max_chars=1500)
# _pp_minimal(res)

Validacion de respuesta más amigable

In [0]:
# """
# response_humanizer.py
# ─────────────────────
# Capa de humanización para las respuestas del RAG híbrido de BIND.
# Toma el dict crudo de `answer_with_rag` y lo reformula usando el LLM
# en un tono conversacional y amigable, manteniendo precisión y trazabilidad.

# Uso en el notebook de validación:
#     from response_humanizer import humanize_response, _pp_humanized

#     res = mod.answer_with_rag(query)
#     _pp_humanized(res)            # pretty print humanizado
#     # o bien:
#     friendly = humanize_response(res)  # solo el texto
# """

# import os
# import json
# import time
# from typing import Optional

# # ──────────────────────────────────────────────────────────────────────────────
# # CONFIG
# # ──────────────────────────────────────────────────────────────────────────────
# LLM_ENDPOINT = os.environ.get("RAG_LLM_ENDPOINT", "databricks-llama-4-maverick")
# MAX_RETRIES  = int(os.environ.get("RAG_MAX_RETRIES", "3"))
# RETRY_SLEEP  = float(os.environ.get("RAG_RETRY_SLEEP_SECS", "1.0"))

# # ──────────────────────────────────────────────────────────────────────────────
# # SYSTEM PROMPT — Personalidad del asistente financiero de BIND
# # ──────────────────────────────────────────────────────────────────────────────
# HUMANIZER_SYSTEM_PROMPT = """\
# Sos un asistente financiero interno del banco BIND. Tu trabajo es tomar una \
# respuesta técnica generada por un sistema RAG y reformularla para que suene \
# natural, clara y amigable — como si un analista senior le estuviera explicando \
# algo a un colega.

# REGLAS:
# 1. PRECISIÓN ANTE TODO: No inventes datos. Usá EXACTAMENTE los números y valores \
#    que aparecen en la respuesta original. Si un dato no está claro, decilo.
# 2. TONO: Profesional pero cercano. Tuteá al usuario. Evitá sonar robótico.
# 3. ESTRUCTURA: Respondé en prosa fluida (no bullet points ni listas numeradas \
#    para el cuerpo). Podés usar párrafos cortos.
# 4. FUENTES: Al final, incluí una sección "📎 Fuentes consultadas:" con las \
#    evidencias numeradas [1], [2], etc. Solo incluí las fuentes que realmente \
#    respaldaron la respuesta.
# 5. CIERRE: Terminá con una frase de apertura tipo: "Si necesitás más detalle o \
#    este valor no es lo que estás buscando, dame un poco más de contexto y lo \
#    revisamos juntos."
# 6. LIMITACIONES: Si la respuesta original indica que no se encontró información, \
#    decilo de forma empática y sugerí cómo reformular la pregunta.
# 7. IDIOMA: Siempre respondé en español (argentino).
# 8. LARGO: Sé conciso. No repitas la misma información con distintas palabras.
# 9. NO incluyas encabezados tipo "RESPUESTA DIRECTA:", "DETALLE:", etc.
# 10. Si hay datos numéricos, formateálos con separador de miles (ej: 50.005 → 50.005).
# """

# # ──────────────────────────────────────────────────────────────────────────────
# # HELPERS
# # ──────────────────────────────────────────────────────────────────────────────

# def _format_evidences(res: dict, max_hits: int = 8) -> str:
#     """Extrae y formatea las evidencias del resultado RAG como texto."""
#     reranked = res.get("reranked_hits") or []
#     cands    = res.get("retrieved_candidates") or []
#     hits     = reranked if reranked else cands

#     if not hits:
#         return "(Sin evidencias disponibles)"

#     lines = []
#     for i, h in enumerate(hits[:max_hits], 1):
#         parts = []
#         file_date = h.get("file_date") or ""
#         path      = h.get("path") or ""
#         page      = h.get("page_num")
#         topic     = h.get("topic") or ""
#         cid       = h.get("chunk_id") or ""

#         if file_date: parts.append(file_date)
#         if path:      parts.append(path)
#         if page is not None: parts.append(f"p.{page}")
#         if topic:     parts.append(f"topic={topic}")

#         loc = " | ".join(parts)
#         lines.append(f"[{i}] {loc}\n    chunk_id: {cid}")

#     return "\n".join(lines)


# def _build_humanizer_prompt(res: dict) -> str:
#     """Construye el prompt de usuario para el LLM humanizador."""
#     query       = res.get("query", "")
#     raw_answer  = res.get("answer", "").strip()
#     evidences   = _format_evidences(res)
#     source_type = res.get("response_source", "desconocido")

#     # Metadata adicional si existe
#     n_cands   = len(res.get("retrieved_candidates") or [])
#     n_rerank  = len(res.get("reranked_hits") or [])

#     return f"""\
# PREGUNTA DEL USUARIO:
# {query}

# RESPUESTA TÉCNICA DEL RAG (fuente: {source_type}):
# {raw_answer}

# EVIDENCIAS DISPONIBLES ({n_cands} candidatos recuperados, {n_rerank} post-reranking):
# {evidences}

# ---
# Reformulá esta respuesta de forma amigable y natural siguiendo tus reglas. \
# Mantené todos los datos numéricos exactos. Incluí las fuentes relevantes al final."""


# def _call_llm(system: str, user: str) -> str:
#     """
#     Llama al LLM endpoint de Databricks via mlflow.deployments.
#     Compatible con el patrón que ya usa el RAG agent.
#     """
#     import mlflow.deployments

#     client = mlflow.deployments.get_deploy_client("databricks")

#     for attempt in range(1, MAX_RETRIES + 1):
#         try:
#             resp = client.predict(
#                 endpoint=LLM_ENDPOINT,
#                 inputs={
#                     "messages": [
#                         {"role": "system", "content": system},
#                         {"role": "user",   "content": user},
#                     ],
#                     "temperature": 0.3,
#                     "max_tokens": 800,
#                 },
#             )

#             # mlflow.deployments devuelve un dict directo
#             if isinstance(resp, dict):
#                 choices = resp.get("choices", [])
#                 if choices:
#                     return choices[0]["message"]["content"].strip()
#                 # Algunos modelos devuelven en "output" o similar
#                 return resp.get("output", str(resp)).strip()

#             return str(resp)

#         except Exception as e:
#             if attempt < MAX_RETRIES:
#                 time.sleep(RETRY_SLEEP * attempt)
#             else:
#                 raise RuntimeError(
#                     f"Error al llamar al LLM humanizador después de {MAX_RETRIES} intentos: {e}"
#                 ) from e


# # ──────────────────────────────────────────────────────────────────────────────
# # API PÚBLICA
# # ──────────────────────────────────────────────────────────────────────────────

# def humanize_response(
#     res: dict,
#     system_prompt: Optional[str] = None,
#     llm_endpoint: Optional[str] = None,
# ) -> str:
#     """
#     Toma el dict crudo de `answer_with_rag()` y devuelve una respuesta
#     amigable y humanizada usando el LLM.

#     Parameters
#     ----------
#     res : dict
#         Output de `mod.answer_with_rag(query)`.
#     system_prompt : str, optional
#         Override del system prompt por defecto.
#     llm_endpoint : str, optional
#         Override del LLM endpoint.

#     Returns
#     -------
#     str
#         Respuesta humanizada lista para mostrar al usuario.
#     """
#     global LLM_ENDPOINT
#     if llm_endpoint:
#         LLM_ENDPOINT = llm_endpoint

#     system = system_prompt or HUMANIZER_SYSTEM_PROMPT
#     user_prompt = _build_humanizer_prompt(res)

#     return _call_llm(system, user_prompt)


# def _pp_humanized(
#     res: dict,
#     show_debug: bool = True,
#     system_prompt: Optional[str] = None,
# ):
#     """
#     Pretty print humanizado para el notebook de validación.
#     Muestra la respuesta amigable + opcionalmente metadata de debug.

#     Parameters
#     ----------
#     res : dict
#         Output de `mod.answer_with_rag(query)`.
#     show_debug : bool
#         Si True, muestra metadata técnica (candidates, hits, etc.).
#     system_prompt : str, optional
#         Override del system prompt.
#     """
#     query = res.get("query", "")

#     print("\n" + "=" * 88)
#     print(f"🔎 PREGUNTA: {query}")
#     print("=" * 88)

#     try:
#         friendly = humanize_response(res, system_prompt=system_prompt)
#         print(f"\n{friendly}")
#     except Exception as e:
#         print(f"\n⚠️  Error al humanizar — mostrando respuesta cruda:\n")
#         print(res.get("answer", "").strip())
#         print(f"\n[Error: {e}]")

#     if show_debug:
#         cands   = res.get("retrieved_candidates") or []
#         reranked = res.get("reranked_hits") or []
#         source   = res.get("response_source", "—")
#         print("\n" + "-" * 88)
#         print(f"📊 Debug: source={source} | candidates={len(cands)} | reranked={len(reranked)}")
#         print("-" * 88)


# def _pp_comparison(res: dict, max_hits: int = 8):
#     """
#     Muestra lado a lado la respuesta cruda vs la humanizada.
#     Útil para validar que la humanización no pierde información.
#     """
#     query = res.get("query", "")

#     print("\n" + "=" * 88)
#     print(f"🔎 PREGUNTA: {query}")
#     print("=" * 88)

#     # --- Respuesta cruda ---
#     print("\n📋 RESPUESTA CRUDA (RAG):")
#     print("-" * 44)
#     print(res.get("answer", "").strip())

#     # --- Evidencias ---
#     print(f"\n📎 EVIDENCIAS:")
#     print("-" * 44)
#     print(_format_evidences(res, max_hits))

#     # --- Respuesta humanizada ---
#     print(f"\n🤖→🧑 RESPUESTA HUMANIZADA:")
#     print("-" * 44)
#     try:
#         friendly = humanize_response(res)
#         print(friendly)
#     except Exception as e:
#         print(f"⚠️  Error: {e}")

#     print("\n" + "=" * 88)

In [0]:
# res = mod.answer_with_rag(query)

# # Opción 1: Solo la respuesta amigable + debug metadata
# _pp_humanized(res)

# # Opción 2: Comparar cruda vs humanizada (para validar que no se pierden datos)
# _pp_comparison(res)

# # Opción 3: Solo obtener el texto humanizado (para integrar en tu pipeline)
# friendly_text = humanize_response(res)

Validación Masiva

In [0]:
# # Set de preguntas para validación
# PREGUNTAS_VALIDACION = [
# "¿Cuál fue el resultado neto de octubre 2025?",
# "¿Cuál fue la variación mensual de los gastos operativos en octubre 2025?",
# "¿Cómo se explica la evolución del ratio de eficiencia en 2025?",
# "¿Cuál fue el peso de los gastos de personal sobre los gastos totales en septiembre 2025?",
# "¿Cuál fue el crecimiento interanual de los ingresos por servicios en octubre 2025?",
# "¿Qué líneas explican la variación de gastos indirectos en septiembre 2025?",
# "¿Cuál fue el crecimiento interanual de los préstamos en $ y en USD?",
# "¿Cómo fue la inflacion interanual de 2025 y la esperada para 2026?",
# "¿Cómo fue la tasa badlar en 2024 vs 2025?",
# "¿Cuál fue el resultado de Banco Industrial en junio 2025 y su diferencia respecto a Banco Santander?",
# "¿Cómo se componen los gastos de septiembre 2025 a nivel banco?",
# "¿Cuál fue la variación interanual de los depósitos en $ para octubre 2025?",
# "¿Cuál fue la variación de ingresos por comisiones para Corporate entre septiembre y octubre 2025?",
# "¿Cuál fue el acumulado de gatos indirectos a octubre 2025?",
# "¿Cuál fue el ROE y ROA del banco en septiembre 2025?",
# "¿Qué margen financiero se obtuvo en octubre 2025 y cómo varió vs septiembre?",
# "¿Cuál fue el resultado antes de impuestos en agosto 2025?",
# "¿Qué impacto tuvo el resultado por títulos en septiembre 2025?",
# "¿Cuál fue la contribución de cada segmento (Retail, Corporate, Pyme) al resultado de octubre 2025?",
# "¿Cuál fue la contribución de cada segmento al resultado de octubre 2025?",
# "¿Cómo evolucionó el spread entre tasa activa y pasiva en 2025?",
# "¿Cómo evolucionaron las altas de clientes en septiembre y octubre 2025?",
# "¿Cuál fue el resultado acumulado a octubre 2025 y cómo se compara con 2024?",
# "¿Como fue los ingresos por comisiones netas para Empresas en marzo de 2025?",
# "¿Cómo fue el crossell para institucional en agosto de 2025?",
# "¿Cómo fue el resultado comercial sin ajuste por inflación en septiembre de 2025 y cuál ha sido el acumulado del año 2025?",
# "¿Qué segmento ha generado más ingresos por margen financiero de préstamos en octubre de 2025?",
# "¿Cual fueron los ingresos por margen financiero de préstamos en octubre de 2025 por segmento?",
# "¿Cual ha sido el spread de préstamos total para baas en agosto, septiembre y octubre de 2025?",
# "¿De cuánto son los dividendos pagados en octubre de 2025?",
# "¿Arma un resumen de cómo estamos en saldos por moneda para octubre de 2025",
# "¿Cómo es el TNA para octubre 2024 para leasing?",
# "¿Calcula la diferencia porcentual entre el Resultado Comercial Neto AxI acumulado para 2025 vs lo presupuestado para 2025",
# "¿Cuál es el dato de previsiones para octubre 2025?",
# "¿Cuáles son los valores de previsiones para octubre 2025 por segmento?",
# "¿Cuáles es el valor de previsiones para octubre 2025 para empresas?",
# "¿Cual es el retorno sobre activos para octubre de 2025?",
# "¿Cual es el retorno sobre patrimonio para octubre de 2025?",
# "¿Como ha sido la evolución del ROE?", #Mejorar obtención de info de gráficos
# "¿Como ha sido la evolución del ROE a lo largo del tiempo?",
# "¿Como ha sido la evolución del ROA?",
# "¿Cómo fue el resultado comercial sin ajuste por inflación en septiembre de 2025 y cuál ha sido el acumulado del año 2025?",
# "¿Cómo es el TNA para octubre 2024 para leasing?",
# "¿Cual fue el resultado comercial neto de corporate para 2025?",
# "¿Cual fue el resultado comercial neto AxI de corporate para 2025?",
# "¿Cuales fueron las previsiones de corporate por cada mes de 2025?",
# "¿Cual es la diferencia entre el acumulado de 2025 real vs el presupuestado?",
# "¿cuál es la variación entre el acumulado del Resultado Comercial Gestion Neto AXI para 2025 vs el presupuestado para 2025?",
# "¿cuál es la diferencia entre el Resultado Comercial Gestion Neto AXI acumulado para 2025 vs lo presupuestado para 2025?",
# "¿Cual es el valor presupuestado de margen financiero de prestamos para 2025?",
# ]

In [0]:
# # =============================================================================
# # CELDA: Validación Batch del RAG - Guardar resultados en Delta y CSV
# # =============================================================================
# # Agregar esta celda al final del notebook RAG_Validation.ipynb
# # Asegúrate de haber ejecutado las celdas anteriores que cargan el módulo (mod)
# # =============================================================================

# from pyspark.sql.types import StructType, StructField, StringType
# from datetime import datetime
# import json

# # -----------------------------------------------------------------------------
# # 1) CONFIGURACIÓN - Modificar según necesidad
# # -----------------------------------------------------------------------------

# # Tabla Delta donde guardar los resultados (catalog.schema.table)
# DELTA_TABLE_NAME = "bind_agent.docs.rag_validation_results"

# # -----------------------------------------------------------------------------
# # 2) FUNCIÓN PARA EXTRAER EVIDENCIAS EN FORMATO LEGIBLE
# # -----------------------------------------------------------------------------

# def extraer_evidencias(res: dict) -> str:
#     """
#     Extrae las evidencias del resultado del RAG y las formatea como string.
#     Combina información de evidence, citations, retrieved_candidates y reranked_hits.
#     """
#     evidencias_parts = []
    
#     # 1) Evidence principal (para respuestas SQL/estructuradas)
#     evidence = res.get("evidence", {})
#     if evidence:
#         source = evidence.get("source", "")
#         source_type = evidence.get("source_type", "")
#         raw_data = evidence.get("raw_data", "")
#         key_points = evidence.get("key_points", [])
        
#         if source or raw_data:
#             ev_str = f"[Fuente: {source} | Tipo: {source_type}]"
#             if key_points:
#                 ev_str += f"\nPuntos clave: {'; '.join(key_points)}"
#             if raw_data:
#                 # Limpiar y truncar raw_data si es muy largo
#                 raw_clean = raw_data.strip().replace('\n', ' | ')[:500]
#                 ev_str += f"\nDatos: {raw_clean}"
#             evidencias_parts.append(ev_str)
    
#     # 2) Citations
#     citations = res.get("citations", [])
#     for i, cit in enumerate(citations[:5], 1):  # Máximo 5 citations
#         cit_source = cit.get("source", "")
#         cit_type = cit.get("source_type", "")
#         cit_content = cit.get("content", "")[:300]  # Truncar contenido
#         if cit_source or cit_content:
#             evidencias_parts.append(f"[Cita {i}: {cit_source} ({cit_type})] {cit_content}")
    
#     # 3) Reranked hits o Retrieved candidates (para respuestas de documentos)
#     hits = res.get("reranked_hits") or res.get("retrieved_candidates") or []
#     for i, hit in enumerate(hits[:5], 1):  # Máximo 5 hits
#         file_date = hit.get("file_date", "")
#         path = hit.get("path", "")
#         page = hit.get("page_num", "")
#         topic = hit.get("topic", "")
#         chunk_id = hit.get("chunk_id", "")
        
#         loc_parts = []
#         if file_date: loc_parts.append(f"Fecha: {file_date}")
#         if path: loc_parts.append(f"Archivo: {path}")
#         if page: loc_parts.append(f"Pág: {page}")
#         if topic: loc_parts.append(f"Tema: {topic}")
#         if chunk_id: loc_parts.append(f"Chunk: {chunk_id}")
        
#         if loc_parts:
#             evidencias_parts.append(f"[Doc {i}] " + " | ".join(loc_parts))
    
#     # 4) Response source
#     response_source = res.get("response_source", "")
#     if response_source:
#         evidencias_parts.append(f"[Fuente de respuesta: {response_source}]")
    
#     # Si no hay evidencias, indicarlo
#     if not evidencias_parts:
#         return "Sin evidencias disponibles"
    
#     return "\n".join(evidencias_parts)

# # -----------------------------------------------------------------------------
# # 3) EJECUTAR VALIDACIÓN BATCH
# # -----------------------------------------------------------------------------

# print(f"Iniciando validación batch de {len(PREGUNTAS_VALIDACION)} preguntas...")
# print("=" * 80)

# resultados = []
# for i, pregunta in enumerate(PREGUNTAS_VALIDACION, 1):
#     print(f"\n[{i}/{len(PREGUNTAS_VALIDACION)}] Procesando: {pregunta[:60]}...")
    
#     try:
#         # Llamar al RAG
#         res = mod.answer_with_rag(pregunta)
        
#         # Extraer campos
#         respuesta = res.get("answer", "").strip()
#         evidencias = extraer_evidencias(res)
        
#         resultados.append({
#             "pregunta": pregunta,
#             "respuesta": respuesta,
#             "evidencias": evidencias
#         })
        
#         print(f"    ✓ Respuesta: {respuesta[:80]}...")
        
#     except Exception as e:
#         print(f"    ✗ Error: {str(e)}")
#         resultados.append({
#             "pregunta": pregunta,
#             "respuesta": f"ERROR: {str(e)}",
#             "evidencias": "Error en procesamiento"
#         })

# print("\n" + "=" * 80)
# print(f"Validación completada. {len(resultados)} resultados obtenidos.")

# # -----------------------------------------------------------------------------
# # 4) CREAR DATAFRAME Y GUARDAR EN DELTA
# # -----------------------------------------------------------------------------

# # Schema de la tabla
# schema = StructType([
#     StructField("pregunta", StringType(), True),
#     StructField("respuesta", StringType(), True),
#     StructField("evidencias", StringType(), True),
# ])

# # Crear DataFrame
# df_resultados = spark.createDataFrame(resultados, schema=schema)

# # Mostrar preview
# print("\nPreview de resultados:")
# df_resultados.show(5, truncate=50)

# # Guardar en tabla Delta (mode="overwrite" para reemplazar, "append" para agregar)
# df_resultados.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .option("overwriteSchema", "true") \
#     .saveAsTable(DELTA_TABLE_NAME)

# print(f"\n✓ Resultados guardados en tabla Delta: {DELTA_TABLE_NAME}")

# # -----------------------------------------------------------------------------
# # 5) GUARDAR EN CSV (en la ubicación del notebook)
# # -----------------------------------------------------------------------------

# # Obtener el path del notebook actual
# notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
# notebook_dir = "/Workspace" + "/".join(notebook_path.rsplit("/", 1)[:-1])

# # Nombre del archivo CSV con timestamp
# timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# csv_filename = f"rag_validation_results_{timestamp}.csv"
# csv_path = f"{notebook_dir}/{csv_filename}"

# # Convertir a Pandas y guardar como CSV
# df_pandas = df_resultados.toPandas()
# df_pandas.to_csv(csv_path, index=False, encoding='utf-8')

# print(f"✓ Resultados guardados en CSV: {csv_path}")

# # -----------------------------------------------------------------------------
# # 6) RESUMEN FINAL
# # -----------------------------------------------------------------------------

# print("\n" + "=" * 80)
# print("RESUMEN DE VALIDACIÓN")
# print("=" * 80)
# print(f"Total de preguntas procesadas: {len(resultados)}")
# print(f"Tabla Delta: {DELTA_TABLE_NAME}")
# print(f"Archivo CSV: {csv_path}")
# print("=" * 80)

# # Mostrar estadísticas
# errores = sum(1 for r in resultados if r["respuesta"].startswith("ERROR"))
# print(f"Exitosos: {len(resultados) - errores}")
# print(f"Con errores: {errores}")